# MemoryLM – Training (Colab / Kaggle)

**Vorher:** Laufzeit → Laufzeittyp ändern → **T4 GPU** (Kaggle: Settings → Accelerator → GPU T4).
Dann Zelle 1 bis 3 ausführen. **Nach einem Abbruch der Sitzung einfach Zelle 1 bis 3 erneut ausführen** –
das Training setzt automatisch am letzten Checkpoint auf Google Drive fort.

### 1 · Google Drive verbinden (Checkpoints, Daten und Logs liegen dort)

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # click "Allow" once
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "NO GPU - change the runtime type!")

### 2 · Code holen und installieren

In [ ]:
REPO = "https://github.com/Flo3200/MemoryLM.git"
CODE = "/content/MemoryLM" if IN_COLAB else "/kaggle/working/MemoryLM_code" if IN_KAGGLE else os.path.abspath("MemoryLM")
if os.path.exists(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

### 3 · Alles ausführen / fortsetzen
Tests → Phase 0 (Daten, Tokenizer, Geschwindigkeitstest) → Phase 1 (Baseline, MemoryLM) → Phase 2 (Rezitation) → Phase 3 (Auswertung).
`DRY_RUN = True` = 10-Minuten-Probelauf.

In [ ]:
PRESET = "quick"      # "full" for the larger model later
DRY_RUN = False
cmd = [sys.executable, "-u", "-m", "memorylm.run_all", "--preset", PRESET] + (["--dry-run"] if DRY_RUN else [])
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")
print("exit code", p.returncode)

### 4 · Optional: fertige Modelle zu Hugging Face hochladen
Vorher Token speichern: Colab → 🔑 *Secrets* (links) → `HF_TOKEN` hinzufügen und Notebook-Zugriff erlauben
(Kaggle: Add-ons → Secrets). **Niemals den Token in eine Zelle schreiben.**

In [ ]:
from memorylm.config import load_config
from memorylm.hub import upload_run
upload_run(load_config(PRESET))